# Check SPI fits on a prepared nClimGrid sample

Compare gamma and Pearson Type III at Timescales 1, 3, and 12 months. From the repository root, run `uv run --group dev scripts/prepare_e2e_inputs.py` once to cache the pinned, reduced NOAA/NCEI nClimGrid example (about 15 MB downloaded), then open this notebook. Its source checksums and subset are recorded in `data/e2e/current/manifest.json`. Start the notebook kernel in `notebooks/` so its default `../data/e2e` path resolves to the prepared data, or set `CLIMATE_INDICES_E2E_DATA` to the prepared-data directory. No source data are downloaded by the notebook.

We inspect only a small central crop: exact per-cell KS p-values are expensive on the full grid. This is a fit audit, not a comparison against independent observations.

In [ ]:
import os
from pathlib import Path

import numpy as np
import xarray as xr

from climate_indices import fit_diagnostics, spi
from climate_indices.indices import Distribution

data_root = Path(os.environ.get('CLIMATE_INDICES_E2E_DATA', '../data/e2e'))
prepared = data_root / 'current' / 'cache_prepared_input.zarr'
if not prepared.exists():
    raise FileNotFoundError(f'{prepared} missing; run uv run --group dev scripts/prepare_e2e_inputs.py')
ds = xr.open_zarr(prepared, consolidated=True)
precip = ds['precip']
if precip.attrs.get('units') != 'mm':
    raise ValueError('Expected monthly precipitation totals in mm')
lat_mid, lon_mid = precip.sizes['lat'] // 2, precip.sizes['lon'] // 2
precip = precip.isel(
    lat=slice(max(0, lat_mid - 2), min(precip.sizes['lat'], lat_mid + 2)),
    lon=slice(max(0, lon_mid - 2), min(precip.sizes['lon'], lon_mid + 2)),
).chunk({'time': -1})
land = precip.notnull().any('time').compute().values
if not land.any():
    raise ValueError('Central crop has no valid precipitation cells; choose another crop')
row, col = np.argwhere(land)[0]
print('Sample cell:', float(precip.lat[row]), float(precip.lon[col]))

In [ ]:
timescales = (1, 3, 12)
distributions = (Distribution.gamma, Distribution.pearson)
fits = {}
pits = {}
for scale in timescales:
    for distribution in distributions:
        options = dict(
            scale=scale, distribution=distribution,
            calibration_year_initial=1981, calibration_year_final=2010,
        )
        fits[scale, distribution] = fit_diagnostics(precip, **options).compute()
        pits[scale, distribution] = spi(
            precip, **options, output_scale='probability',
        ).sel(lat=precip.lat[row], lon=precip.lon[col]).compute()
        assert fits[scale, distribution].sizes['month'] == 12
        assert pits[scale, distribution].attrs['climate_indices_variant'] == 'probability'
print('Fit families used:', {
    (scale, distribution.value): np.unique(fits[scale, distribution].distribution_used.values).tolist()
    for scale in timescales for distribution in distributions
})

## Probability-scale histograms

Use the same 1981–2010 Calibration Period as the fit. Exact zeros (and Pearson trace values below 0.0005) usually create a spike at `p0` under classic zero placement; gamma resets an all-zero calibration step's `p0` to 0, and Pearson support-limit masking can override the score. A uniform histogram is not expected in a zero-inflated month. The bounded or normal output scale is **not** a PIT histogram.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 2, figsize=(10, 9), sharex=True, sharey=True)
for i, scale in enumerate(timescales):
    for j, distribution in enumerate(distributions):
        values = pits[scale, distribution].sel(time=slice('1981', '2010')).values
        axes[i, j].hist(values[np.isfinite(values)], bins=np.linspace(0, 1, 11))
        used = fits[scale, distribution].distribution_used.isel(lat=row, lon=col).item()
        axes[i, j].set(title=f'SPI-{scale} {distribution.value} (used {used})', xlabel='PIT', ylabel='Count')
fig.tight_layout()
plt.show()

## January KS p-values and probability of zero

Maps cover the same crop, not the entire nClimGrid domain. `n_valid` counts non-zero calibration values entering the KS test. The test omits zero and missing values; fitted parameters use the same calibration record (including zeros for Pearson), so a low p-value flags a fit to investigate rather than providing a calibrated hypothesis test. Missing p-values mean the test could not assess the fit. Pearson panels mask cells whose `distribution_used` reports gamma fallback; inspect that variable to see where fallback occurred.

In [ ]:
for variable, title in [('ks_p_value', 'January KS p-value'), ('prob_zero', 'January probability of zero')]:
    fig, axes = plt.subplots(3, 2, figsize=(10, 10))
    for i, scale in enumerate(timescales):
        for j, distribution in enumerate(distributions):
            fit = fits[scale, distribution]
            fit[variable].sel(month=1).where(
                fit.distribution_used == distribution.value
            ).plot(
                ax=axes[i, j], vmin=0, vmax=1, cmap='viridis',
            )
            axes[i, j].set_title(f'SPI-{scale} {distribution.value}: {title}')
    fig.tight_layout()
    plt.show()
ds.close()